# Register WNV Assistant Model

Logs the agent as an MLflow model and registers it to Unity Catalog.

In [ ]:
# Databricks Git folders are available through the /Workspace filesystem.
# This notebook lives at <repo>/notebooks/register_model.ipynb.
from pathlib import Path

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
repo_workspace_path = notebook_path.rsplit('/', 2)[0]
repo_fs_path = (
    repo_workspace_path
    if repo_workspace_path.startswith('/Workspace/')
    else f'/Workspace{repo_workspace_path}'
)
if not (Path(repo_fs_path) / 'pyproject.toml').is_file():
    raise FileNotFoundError(f'Could not find project root at {repo_fs_path}')
print(f'Repo filesystem path: {repo_fs_path}')

# Install the project as a regular package. Do not mutate sys.path.
%pip install {repo_fs_path} databricks-sql-connector mlflow
dbutils.library.restartPython()

In [ ]:
import mlflow

# Register models in the Unity Catalog Model Registry, not the legacy workspace registry.
mlflow.set_registry_uri("databricks-uc")
from wnv_assistant.serving import log_model

# Log the model
model_uri = log_model("/tmp/wnv-model")
print(f"Model logged: {model_uri}")

In [ ]:
# Register to Unity Catalog
model_version = mlflow.register_model(
    model_uri=model_uri,
    name="eliao.wnv_demo.wnv_assistant",
)
print(f"Registered: {model_version}")

In [ ]:
# Test the logged model
import mlflow.pyfunc

model = mlflow.pyfunc.load_model(model_uri)
result = model.predict({"question": "Show top 5 counties by mosquito activity in 2022"})
print(result)